# [6154.71] NeuroGolf Manual ONNX Rewrites + Hand-Built Solvers

**Current public score target: 6154.71**. This is the same public notebook ID as the original 43-vote `[6042.85]` write-up, updated so a fresh Kaggle run emits the clean v205 manual-only artifact.

Scope of this update:

- Keep the original 6042.85 hand-built solver explanation below.
- Publish the accepted v205 bundle built from hand-written solvers, task-local manual rewrites, and accepted-model structural micro-optimizations.
- Exclude later public-source blends and swaps from `good-solution.zip`, Anazemcev, Massimiliano Ghiotto, Biohack44, and derived blend notebooks.

The v205 artifact is attached as a small public dataset so the notebook can rerun quickly and rescore transparently. The final cell overwrites `/kaggle/working/submission.zip` with that manual-only 400-task ONNX bundle.


## Manual-only v205 provenance

This score came from the manual rewrite ladder after the original three hand-builds. The compressed provenance table is included in the attached dataset and printed here before the original notebook body.


In [ ]:
from pathlib import Path
import csv

manual_dataset = Path('/kaggle/input/neurogolf-manual-rewrites-v205')
provenance_path = manual_dataset / 'manual_rewrite_provenance.csv'

if provenance_path.exists():
    with provenance_path.open(newline='') as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        print(f"{row['version']:>4} | {row['public_lb']:>7} | {row['change']}")
else:
    print('manual rewrite provenance dataset not attached yet')


## The negative result that paid for the methodology

Before describing the wins, the loss is worth recording because it changes the rules of which
experiments are worth running.

**v16: cross-bundle swap, predicted +116 LB, grader -308 LB.** The 6029 base bundle ships
expensive (~1MB) hand-built solvers for tasks like 084, 153, 200, 285. The local cost ranking
said the older afr1ste 5689 or konbu17 versions of those tasks were 100x cheaper and should
produce a +116 LB jump if swapped in. The grader returned **5720.77** instead of the predicted
6145.73. Of the ~350 LB of original score in the 33 swapped tasks, only ~42 LB retained on
grader - most cheaper-public-versions scored 0 on the hidden private benchmark.

The lesson: the bundle author already tested those cheap versions and rejected them. The bundle's
expensive solvers exist precisely *because* the cheaper alternatives fail the hidden test. Local
arc-gen + train + test verification is **not a proxy** for the hidden split when swapping bytes
built for a different purpose.

**The flip side, validated three times in this notebook**: hand-built solvers that derive the
task's TRUE GENERAL RULE from train+test+arc-gen and implement it from scratch DO pass the
hidden test. The local oracle becomes grader-faithful for such solvers. Three submissions, three
exact matches to within 0.01 LB.


In [ ]:
import subprocess, sys
try:
    import onnxruntime as _ort  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'])
    import onnxruntime as _ort  # noqa: F401


In [ ]:
import json
import math
import os
import shutil
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
from onnx import TensorProto
from onnx import helper as oh
from onnx import numpy_helper as onh
from onnx import version_converter

NUM_TASKS = 400
INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
WORKING.mkdir(parents=True, exist_ok=True)

OUTPUT_ZIP = WORKING / 'submission.zip'

# 6029 base bundle - 400 task ONNX, will be used verbatim for 397 of 400 tasks
JSRDCHT_DATASET = 'neurogolf-6029-submission-bundle'

# Competition data (task001.json .. task400.json) - used only for verifying hand-builds
COMP_DIR = Path('/kaggle/input/competitions/neurogolf-2026')
if not COMP_DIR.exists():
    COMP_DIR = Path('/kaggle/input/neurogolf-2026')

def find_onnx_root(slug):
    for entry in INPUT_ROOT.rglob('task001.onnx'):
        if slug in str(entry):
            return entry.parent
    raise FileNotFoundError('Could not locate task ONNX root for ' + slug)

jsrdcht_dir = find_onnx_root(JSRDCHT_DATASET)
print('jsrdcht 6029 bundle :', jsrdcht_dir)
print('competition data    :', COMP_DIR)


## The local oracle: cost prediction matches grader to the hundredth

The grader scores each task ONNX with `cost = memory + params` and converts to points via
`max(1, 25 - ln(max(1, cost)))`. Both terms are computable locally:

- `params` is the element count of every initializer plus every Constant node value
- `memory` is the sum over intermediate tensors of `max(static_inferred_size, ORT_runtime_size)`
  computed from a single ORT profile trace

For hand-built ONNX where the rule is derived from the task's true semantics, the local cost
matches the grader's cost essentially to the byte. That has held for each hand-build added to
this notebook (column 'Predicted' vs column 'Grader' in the intro table).


In [ ]:
def encode_grid(grid):
    arr = np.array(grid, dtype=np.int32)
    h, w = arr.shape
    t = np.zeros((1, 10, 30, 30), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            v = int(arr[r, c])
            if 0 <= v < 10:
                t[0, v, r, c] = 1.0
    return t

def calculate_params(model):
    n = 0
    for init in model.graph.initializer:
        n += int(math.prod(init.dims)) if init.dims else 1
    for si in model.graph.sparse_initializer:
        n += int(math.prod(si.values.dims)) if si.values.dims else 1
    for node in model.graph.node:
        if node.op_type != 'Constant':
            continue
        for attr in node.attribute:
            if attr.name == 'value':
                n += int(math.prod(attr.t.dims)) if attr.t.dims else 1
            elif attr.name == 'value_floats':
                n += len(attr.floats)
            elif attr.name == 'value_ints':
                n += len(attr.ints)
    return n

def calculate_memory(model_path, examples, n_runs=3):
    model = onnx.load(str(model_path))
    onnx.checker.check_model(model, full_check=True)
    graph = onnx.shape_inference.infer_shapes(model, strict_mode=True).graph

    tensor_dtype = {}
    tensor_static = {}
    for vi in list(graph.input) + list(graph.value_info) + list(graph.output):
        if not vi.type.HasField('tensor_type'):
            continue
        shape = vi.type.tensor_type.shape
        if not shape.dim:
            continue
        dims = []
        ok = True
        for d in shape.dim:
            if not d.HasField('dim_value') or d.dim_value <= 0:
                ok = False
                break
            dims.append(d.dim_value)
        if not ok:
            continue
        np_dt = onnx.helper.tensor_dtype_to_np_dtype(vi.type.tensor_type.elem_type)
        tensor_dtype[vi.name] = np_dt
        tensor_static[vi.name] = int(np.prod(dims)) * np.dtype(np_dt).itemsize

    node_outputs = {n.name: list(n.output) for n in graph.node}

    opts = ort.SessionOptions()
    opts.enable_profiling = True
    opts.log_severity_level = 3
    sess = ort.InferenceSession(str(model_path), opts, providers=['CPUExecutionProvider'])
    for p in examples[:n_runs]:
        _ = sess.run(['output'], {'input': encode_grid(p['input'])})
    trace_path = sess.end_profiling()
    with open(trace_path) as f:
        trace = json.load(f)
    os.remove(trace_path)

    tensor_runtime = {}
    for event in trace:
        if event.get('cat') != 'Node' or 'args' not in event:
            continue
        if 'output_type_shape' not in event['args']:
            continue
        node_name = event.get('name', '').replace('_kernel_time', '')
        if node_name not in node_outputs:
            continue
        outs = node_outputs[node_name]
        for i, shape_dict in enumerate(event['args']['output_type_shape']):
            if i >= len(outs):
                continue
            name = outs[i]
            if name not in tensor_dtype:
                continue
            itemsize = np.dtype(tensor_dtype[name]).itemsize
            sz = itemsize * sum(int(math.prod(dims)) for dims in shape_dict.values())
            tensor_runtime[name] = max(tensor_runtime.get(name, 0), sz)

    total = 0
    for name, static in tensor_static.items():
        if name in ('input', 'output'):
            continue
        total += max(static, tensor_runtime.get(name, 0))
    return total

def cost_and_score(model_path, examples):
    mem = calculate_memory(model_path, examples)
    model = onnx.load(str(model_path))
    params = calculate_params(model)
    cost = mem + params
    score = max(1.0, 25.0 - math.log(max(1.0, cost)))
    return cost, score, mem, params

def verify(model_path, examples):
    sess = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
    n_pass = 0
    for p in examples:
        try:
            out = sess.run(['output'], {'input': encode_grid(p['input'])})[0]
            tgt = encode_grid(p['output']) > 0.0
            if np.array_equal(out > 0.0, tgt):
                n_pass += 1
        except Exception:
            pass
    return n_pass, len(examples)

def load_examples(task_id):
    p = COMP_DIR / f'task{task_id:03d}.json'
    if p.exists():
        return json.load(p.open())
    for c in COMP_DIR.rglob(f'task{task_id:03d}.json'):
        return json.load(c.open())
    raise FileNotFoundError(f'task{task_id:03d}.json')


## Hand-build 1: task277 (8-connected components, label unique-size with color 2)

**Rule** (verified 266/266 across train + test + arc-gen): the input has exactly three
connected components of color 8 (8-connectivity). The component whose cell-count is unique
among the three is recolored 2; the other two are recolored 1; color 0 passes through.

**ONNX strategy** without the banned ops (Loop, Scan, NonZero, Unique, Compress):

1. Initialize a per-cell label tensor with `row*30+col` for color-8 cells and a sentinel value
   elsewhere. Negate the tensor so we can use MaxPool 3x3 to compute MinPool via the
   `-MaxPool(-x)` trick.
2. Iterate the negated-MaxPool five times, re-masking after each iteration so that color-0
   cells always reset to the sentinel. After convergence every color-8 cell holds the
   minimum cell index in its 8-connected component.
3. Extract the three component IDs by `ReduceMin` -> mask off the cells of that component ->
   `ReduceMin` again, twice. Three IDs in scalar form, no Unique op.
4. Count cells per component via `ReduceSum` of each component's mask.
5. Determine which count is unique (the count that is different from both other counts) via
   `Equal` + `Not` + `And`.
6. Build the output channel-by-channel: channel 0 from input, channel 1 from the union of
   non-unique component masks, channel 2 from the unique component mask, all other channels zero.

Local cost: 120K bytes. Grader: cost 120K, score 13.30. Net contribution: **+13.30 LB**.


In [ ]:
"""Hand-build task277: label 8-pixel connected components 1 or 2 based on whether
the component's cell count is unique among the 3 components in the grid.

Rule (verified on 266/266 pairs): 8-connected components of color-8 pixels;
component with unique cell-count gets color 2, others get color 1. Color 0
passes through.

Strategy without banned ops (Loop/Scan/NonZero/Unique/Compress):
  1. mask = input channel 8; ch0 = input channel 0
  2. Label propagation: init labels with arange(900) where mask=1 (BIG elsewhere),
     iterate 5 times with MinPool (via -MaxPool of -x), re-mask after each pool.
  3. Extract 3 component IDs by ReduceMin, then mask-out and repeat.
  4. Count cells in each component via ReduceSum of equality masks.
  5. Uniqueness: count_i is unique iff (count_i != count_j) AND (count_i != count_k).
  6. Per-cell output value = 2 if its component count is unique else 1.
  7. Build 10-channel output: ch0 = original ch0, ch1 = is_label_1, ch2 = is_label_2,
     ch3..9 = 0.

Always-3-components assumption holds across 266/266 task277 pairs.
"""
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, helper as oh, numpy_helper as onh


ROOT = Path("/kaggle/working")
OUT_PATH = ROOT / "task277.onnx"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

BIG = 999.0  # > 30*30 = 900 max index; used as "unmarked" sentinel
SHAPE_4D = [1, 1, 30, 30]
PAD_TENSOR = np.array([0, 0, 1, 1, 0, 0, 1, 1], dtype=np.int64)  # 3x3 kernel pads by 1


def fp(name, value, shape=None):
    arr = np.array(value, dtype=np.float32)
    if shape is not None:
        arr = arr.reshape(shape)
    return onh.from_array(arr, name=name)


def i64(name, value):
    return onh.from_array(np.array(value, dtype=np.int64), name=name)


def build():
    initializers = []

    # Negated index tensor [1,1,30,30] with -(row*30+col) at each cell.
    # We keep labels in negated form throughout label-propagation to save Neg ops.
    neg_idx = -np.arange(900, dtype=np.float32).reshape(1, 1, 30, 30)
    initializers.append(onh.from_array(neg_idx, name="neg_idx"))

    initializers.append(fp("BIG", BIG))
    initializers.append(fp("NEG_BIG", -BIG))
    initializers.append(fp("ONE", 1.0))
    initializers.append(fp("TWO", 2.0))
    initializers.append(fp("ZERO", 0.0))
    initializers.append(onh.from_array(PAD_TENSOR, name="pads_hw"))

    # Slice indices for channel selection
    initializers.append(i64("slice_starts_ch0", [0, 0, 0, 0]))
    initializers.append(i64("slice_ends_ch0", [1, 1, 30, 30]))
    initializers.append(i64("slice_starts_ch8", [0, 8, 0, 0]))
    initializers.append(i64("slice_ends_ch8", [1, 9, 30, 30]))
    initializers.append(i64("slice_axes", [0, 1, 2, 3]))
    initializers.append(i64("reduce_axes_all", [0, 1, 2, 3]))

    nodes = []

    # ---- Step 1: extract mask (channel 8) and ch0 (channel 0) ----
    nodes.append(oh.make_node("Slice", ["input", "slice_starts_ch0", "slice_ends_ch0", "slice_axes"], ["ch0"], name="slice_ch0"))
    nodes.append(oh.make_node("Slice", ["input", "slice_starts_ch8", "slice_ends_ch8", "slice_axes"], ["mask"], name="slice_ch8"))
    # mask_bool: re-used by all Where re-masks below
    nodes.append(oh.make_node("Greater", ["mask", "ZERO"], ["mask_bool"], name="mask_bool"))

    # Keep labels NEGATED throughout the loop to save per-iter Neg ops.
    # nlabels_0 = Where(mask_bool, -idx, NEG_BIG). We store -idx as an initializer.
    nodes.append(oh.make_node("Where", ["mask_bool", "neg_idx", "NEG_BIG"], ["nlabels_0"], name="nlabels_init"))

    # ---- Step 2: 5 iterations of MaxPool 3x3 on NEGATED labels (equivalent to MinPool) ----
    prev = "nlabels_0"
    for it in range(5):
        padded = f"pad_{it}"
        nodes.append(oh.make_node("Pad", [prev, "pads_hw", "NEG_BIG"], [padded], mode="constant", name=f"pad_{it}"))
        pooled = f"pooled_{it}"
        nodes.append(oh.make_node("MaxPool", [padded], [pooled],
                                  kernel_shape=[3, 3], strides=[1, 1], pads=[0, 0, 0, 0],
                                  name=f"pool_{it}"))
        nxt = f"nlabels_{it+1}"
        nodes.append(oh.make_node("Where", ["mask_bool", pooled, "NEG_BIG"], [nxt], name=f"remask_{it}"))
        prev = nxt
    # Un-negate once at the end to get final labels for ReduceMin / Equal
    nodes.append(oh.make_node("Neg", [prev], ["labels"], name="labels_final"))
    labels = "labels"  # shape [1,1,30,30]

    # ---- Step 3: extract 3 component IDs via successive ReduceMin + mask-out ----
    # c1 = ReduceMin(labels, all)  -- shape [1,1,1,1]
    nodes.append(oh.make_node("ReduceMin", [labels], ["c1"], axes=[0, 1, 2, 3], keepdims=1, name="rmin_c1"))
    nodes.append(oh.make_node("Equal", [labels, "c1"], ["mask_c1_b"], name="eq_c1"))
    nodes.append(oh.make_node("Cast", ["mask_c1_b"], ["mask_c1"], to=TensorProto.FLOAT, name="cast_c1"))

    # labels_sup1 = where(mask_c1_b, BIG, labels)
    nodes.append(oh.make_node("Where", ["mask_c1_b", "BIG", labels], ["labels_sup1"], name="suppress_c1"))

    nodes.append(oh.make_node("ReduceMin", ["labels_sup1"], ["c2"], axes=[0, 1, 2, 3], keepdims=1, name="rmin_c2"))
    nodes.append(oh.make_node("Equal", [labels, "c2"], ["mask_c2_b"], name="eq_c2"))
    nodes.append(oh.make_node("Cast", ["mask_c2_b"], ["mask_c2"], to=TensorProto.FLOAT, name="cast_c2"))
    nodes.append(oh.make_node("Where", ["mask_c2_b", "BIG", "labels_sup1"], ["labels_sup2"], name="suppress_c2"))

    nodes.append(oh.make_node("ReduceMin", ["labels_sup2"], ["c3"], axes=[0, 1, 2, 3], keepdims=1, name="rmin_c3"))
    nodes.append(oh.make_node("Equal", [labels, "c3"], ["mask_c3_b"], name="eq_c3"))
    nodes.append(oh.make_node("Cast", ["mask_c3_b"], ["mask_c3"], to=TensorProto.FLOAT, name="cast_c3"))

    # ---- Step 4: counts per component ----
    nodes.append(oh.make_node("ReduceSum", ["mask_c1", "reduce_axes_all"], ["count1"], keepdims=1, name="rsum_c1"))
    nodes.append(oh.make_node("ReduceSum", ["mask_c2", "reduce_axes_all"], ["count2"], keepdims=1, name="rsum_c2"))
    nodes.append(oh.make_node("ReduceSum", ["mask_c3", "reduce_axes_all"], ["count3"], keepdims=1, name="rsum_c3"))

    # ---- Step 5: uniqueness check ----
    # ne_ij = NOT(count_i == count_j)
    nodes.append(oh.make_node("Equal", ["count1", "count2"], ["eq12"], name="eq12"))
    nodes.append(oh.make_node("Equal", ["count1", "count3"], ["eq13"], name="eq13"))
    nodes.append(oh.make_node("Equal", ["count2", "count3"], ["eq23"], name="eq23"))
    nodes.append(oh.make_node("Not", ["eq12"], ["ne12"], name="ne12"))
    nodes.append(oh.make_node("Not", ["eq13"], ["ne13"], name="ne13"))
    nodes.append(oh.make_node("Not", ["eq23"], ["ne23"], name="ne23"))
    nodes.append(oh.make_node("And", ["ne12", "ne13"], ["unique1"], name="unique1"))
    nodes.append(oh.make_node("And", ["ne12", "ne23"], ["unique2"], name="unique2"))
    nodes.append(oh.make_node("And", ["ne13", "ne23"], ["unique3"], name="unique3"))

    # ---- Step 6: per-component label value (2.0 if unique else 1.0) ----
    nodes.append(oh.make_node("Where", ["unique1", "TWO", "ONE"], ["lv1"], name="lv1"))
    nodes.append(oh.make_node("Where", ["unique2", "TWO", "ONE"], ["lv2"], name="lv2"))
    nodes.append(oh.make_node("Where", ["unique3", "TWO", "ONE"], ["lv3"], name="lv3"))

    # ---- Step 7: per-cell output value = sum of mask_ci * lv_i ----
    nodes.append(oh.make_node("Mul", ["mask_c1", "lv1"], ["m1l"], name="m1l"))
    nodes.append(oh.make_node("Mul", ["mask_c2", "lv2"], ["m2l"], name="m2l"))
    nodes.append(oh.make_node("Mul", ["mask_c3", "lv3"], ["m3l"], name="m3l"))
    nodes.append(oh.make_node("Sum", ["m1l", "m2l", "m3l"], ["val"], name="sum_vals"))  # shape [1,1,30,30]

    # ---- Step 8: build 10 output channels ----
    # is_1 = (val == 1).cast(fp32)
    nodes.append(oh.make_node("Equal", ["val", "ONE"], ["is_1_b"], name="is1"))
    nodes.append(oh.make_node("Cast", ["is_1_b"], ["is_1"], to=TensorProto.FLOAT, name="cast_is1"))
    nodes.append(oh.make_node("Equal", ["val", "TWO"], ["is_2_b"], name="is2"))
    nodes.append(oh.make_node("Cast", ["is_2_b"], ["is_2"], to=TensorProto.FLOAT, name="cast_is2"))

    # zero_channel = ch0 * 0 (cheap way to get [1,1,30,30] of zeros)
    nodes.append(oh.make_node("Mul", ["ch0", "ZERO"], ["zero_ch"], name="zero_ch"))

    # Concat [ch0, is_1, is_2, zero, zero, zero, zero, zero, zero, zero] on axis 1
    concat_inputs = ["ch0", "is_1", "is_2"] + ["zero_ch"] * 7
    nodes.append(oh.make_node("Concat", concat_inputs, ["output"], axis=1, name="concat_out"))

    input_vi = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
    output_vi = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])
    graph = oh.make_graph(nodes=nodes, name="task277",
                          inputs=[input_vi], outputs=[output_vi],
                          initializer=initializers)
    opset = oh.make_opsetid("", 14)
    model = oh.make_model(graph, opset_imports=[opset], ir_version=8)
    model.producer_name = ""
    onnx.checker.check_model(model, full_check=True)
    onnx.save(model, str(OUT_PATH))
    return model

_t277 = build()
print(f'task277 built: {OUT_PATH.stat().st_size} bytes, {len(_t277.graph.node)} nodes')


In [ ]:
_t277_examples = load_examples(277)
_t277_ex = _t277_examples['train'] + _t277_examples['test'] + _t277_examples.get('arc-gen', [])
_t277_pass, _t277_total = verify(WORKING / 'task277.onnx', _t277_ex)
_t277_cost, _t277_score, _, _ = cost_and_score(WORKING / 'task277.onnx', _t277_ex)
print(f'task277: verify {_t277_pass}/{_t277_total}, cost {_t277_cost}, predicted score {_t277_score:.3f}')


## Hand-build 2: task330 (size-threshold recolor via ScatterND-add histogram)

**Rule** (verified 266/266): connected components of color 5 (8-connectivity) get recolored
by their cell count. Components of exactly 6 cells become color 2; all other sizes become
color 1.

**The new tool added to the kit**: opset 17 `ScatterND` with `reduction='add'` lets us build a
histogram of per-component counts in `O(cells)` memory. The pattern (since validated
grader-exact in v18):

```
labels = label_propagation(mask)            # per-cell component min-id
labels_idx = Unsqueeze(Cast(labels, INT64), axes=[4])   # [1,1,30,30,1]
hist = ConstantOfShape([1000])                          # zeros, size 1000 so BIG sentinel
                                                        # for color-0 cells lands in unused bucket
hist = ScatterND(hist, labels_idx, mask, reduction='add')   # hist[id] += mask
count_per_cell = Gather(hist, labels, axis=0)           # [1,1,30,30]
```

After that, `output_ch_2 = Where(count == 6, mask, 0)` and `output_ch_1 = Where(count == 6, 0, mask)`
split the mask cells by their component size.

**Local cost**: 62K bytes (the bundle's task330 was 96K). Grader: cost 62K, score 13.97. Net
contribution: **+0.44 LB**.


In [ ]:
"""task330 ONNX builder.

Rule (verified 266/266): for each connected component of color-5 (8-conn),
recolor cells to 2 if component has exactly 6 cells, else to 1.

Strategy without banned ops:
  1. Extract mask = ch_5. Label propagation via 5 iters of negated-MaxPool 3x3.
  2. Compute per-cell component min-id label.
  3. Build histogram via ScatterND with reduction='add' (opset 16+):
     hist[label_at_cell] += 1 (for mask=1 cells).
  4. Gather each cell's component count from hist.
  5. color = 2 if count == 6 else 1, only for mask=1 cells.
  6. Build output channels: ch_0 preserved, ch_1 = (mask AND count != 6), ch_2 = (mask AND count == 6).
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, helper as oh, numpy_helper as onh

ROOT = Path("/kaggle/working")
OUT_PATH = ROOT / "task330.onnx"

BIG = 999.0
PAD_TENSOR = np.array([0, 0, 1, 1, 0, 0, 1, 1], dtype=np.int64)


def fp(name, value):
    return onh.from_array(np.array(value, dtype=np.float32), name=name)


def i64(name, value):
    return onh.from_array(np.array(value, dtype=np.int64), name=name)


def build():
    inits = []

    # Negated index tensor [1,1,30,30] fp16 to halve propagation memory.
    neg_idx = -np.arange(900, dtype=np.float16).reshape(1, 1, 30, 30)
    inits.append(onh.from_array(neg_idx, name="neg_idx"))

    inits.append(onh.from_array(np.array(-BIG, dtype=np.float16), name="NEG_BIG"))
    inits.append(fp("BIG", BIG))
    inits.append(fp("ZERO", 0.0))
    inits.append(fp("ONE", 1.0))
    inits.append(fp("TWO", 2.0))
    inits.append(fp("SIX", 6.0))

    inits.append(onh.from_array(PAD_TENSOR, name="pads_hw"))

    # Slice indices for channels
    inits.append(i64("starts_ch0", [0, 0, 0, 0]))
    inits.append(i64("ends_ch0", [1, 1, 30, 30]))
    inits.append(i64("starts_ch5", [0, 5, 0, 0]))
    inits.append(i64("ends_ch5", [1, 6, 30, 30]))
    inits.append(i64("axes_all", [0, 1, 2, 3]))

    # Hist size 1000 â€” accommodates the BIG=999 sentinel for mask=0 cells in an
    # unused bucket, avoiding the clamp Where.
    inits.append(i64("hist_shape", [1000]))
    inits.append(i64("axes_unsqueeze_last", [4]))
    # Static zero channel [1,1,30,30] for output-channel padding (saves 3600 bytes vs computed)
    inits.append(onh.from_array(np.zeros((1, 1, 30, 30), dtype=np.float32), name="zero_ch"))

    nodes = []

    # ---- Step 1: extract channels ----
    nodes.append(oh.make_node("Slice", ["input", "starts_ch0", "ends_ch0", "axes_all"], ["ch0"], name="slice_ch0"))
    nodes.append(oh.make_node("Slice", ["input", "starts_ch5", "ends_ch5", "axes_all"], ["mask"], name="slice_ch5"))
    nodes.append(oh.make_node("Greater", ["mask", "ZERO"], ["mask_bool"], name="mask_bool"))

    # ---- Step 2: label propagation (4 iterations, verified max graph dist = 4) ----
    # Use MaxPool's internal padding (auto -inf for max), avoiding explicit Pad nodes.
    nodes.append(oh.make_node("Where", ["mask_bool", "neg_idx", "NEG_BIG"], ["nlabels_0"], name="nlabels_init"))
    prev = "nlabels_0"
    for it in range(4):
        pooled = f"pooled_{it}"
        nodes.append(oh.make_node("MaxPool", [prev], [pooled],
                                  kernel_shape=[3, 3], strides=[1, 1], pads=[1, 1, 1, 1],
                                  name=f"pool_{it}"))
        nxt = f"nlabels_{it+1}"
        nodes.append(oh.make_node("Where", ["mask_bool", pooled, "NEG_BIG"], [nxt], name=f"remask_{it}"))
        prev = nxt
    # Un-negate to get positive labels (in [0, 899] for mask=1 cells, BIG for mask=0)
    nodes.append(oh.make_node("Neg", [prev], ["labels"], name="labels_final"))

    # ---- Step 3: histogram via ScatterND-with-add ----
    # Cast labels to int64 (ScatterND indices require int64). mask=0 cells have value
    # BIG=999, which lands in the unused bucket of the 1000-sized histogram.
    nodes.append(oh.make_node("Cast", ["labels"], ["labels_i64"], to=TensorProto.INT64, name="cast_labels"))
    # Add trailing index dim: [1,1,30,30] -> [1,1,30,30,1] for ScatterND
    nodes.append(oh.make_node("Unsqueeze", ["labels_i64", "axes_unsqueeze_last"], ["labels_idx"], name="unsq_idx"))
    # Initialize hist [1000] = zeros
    nodes.append(oh.make_node("ConstantOfShape", ["hist_shape"], ["hist_init"],
                              value=onh.from_array(np.array([0.0], dtype=np.float32)),
                              name="hist_init"))
    # ScatterND with reduction='add': hist[label[i,j,k,l]] += mask[i,j,k,l]
    nodes.append(oh.make_node("ScatterND", ["hist_init", "labels_idx", "mask"],
                              ["hist"], reduction="add", name="scatter_hist"))

    # ---- Step 4: gather per-cell count directly (no flatten) ----
    nodes.append(oh.make_node("Gather", ["hist", "labels_i64"], ["count"], axis=0, name="gather_count"))

    # ---- Step 5: build output channels directly from eq_6 + mask ----
    nodes.append(oh.make_node("Equal", ["count", "SIX"], ["eq_6"], name="eq_6"))
    # ch2_out = mask where size==6 else 0
    nodes.append(oh.make_node("Where", ["eq_6", "mask", "ZERO"], ["ch2_out"], name="ch2_out"))
    # ch1_out = mask where size!=6 else 0
    nodes.append(oh.make_node("Where", ["eq_6", "ZERO", "mask"], ["ch1_out"], name="ch1_out"))
    # ch_0_out = ch0 (cells of color 0 stay color 0)
    # Concat 10 channels: [ch0, ch1, ch2, zero, zero, zero, zero, zero, zero, zero]
    concat_inputs = ["ch0", "ch1_out", "ch2_out"] + ["zero_ch"] * 7
    nodes.append(oh.make_node("Concat", concat_inputs, ["output"], axis=1, name="concat_out"))

    input_vi = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
    output_vi = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])
    graph = oh.make_graph(nodes=nodes, name="task330",
                          inputs=[input_vi], outputs=[output_vi],
                          initializer=inits)
    # opset 17 (grader accepts; bundle's task277 uses 17)
    opset = oh.make_opsetid("", 17)
    model = oh.make_model(graph, opset_imports=[opset], ir_version=8)
    model.producer_name = ""
    onnx.checker.check_model(model, full_check=True)
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    onnx.save(model, str(OUT_PATH))
    return model

_t330 = build()
print(f'task330 built: {OUT_PATH.stat().st_size} bytes, {len(_t330.graph.node)} nodes')


In [ ]:
_t330_examples = load_examples(330)
_t330_ex = _t330_examples['train'] + _t330_examples['test'] + _t330_examples.get('arc-gen', [])
_t330_pass, _t330_total = verify(WORKING / 'task330.onnx', _t330_ex)
_t330_cost, _t330_score, _, _ = cost_and_score(WORKING / 'task330.onnx', _t330_ex)
print(f'task330: verify {_t330_pass}/{_t330_total}, cost {_t330_cost}, predicted score {_t330_score:.3f}')


## Hand-build 3: task364 (topology-based recolor by endpoints + turns)

**Rule** (verified 266/266): connected components of color 3 (4-connectivity) get recolored by
their graph topology. For each component, count two structural features:

- **endpoints**: cells with exactly one same-color 4-neighbor (leaves of the tree)
- **turns**: cells with two same-color 4-neighbors where those two neighbors are NOT collinear
  (one horizontal + one vertical, i.e. the cell sits at a 90 degree bend)

Decision tree:
- if endpoints >= 3 -> recolor to **2** (branching tree, e.g. T-shape or plus)
- elif turns >= 2 -> recolor to **6** (path with two or more bends, e.g. C-shape or zigzag)
- else -> recolor to **1** (straight or single-bend path, e.g. L)

**ONNX strategy**: each cell's 4-neighbor presence is a Pad+Slice shift. Sum up/down for
`n_vert`, left/right for `n_horiz`, total for `deg_4`. Endpoints are `mask AND deg_4 == 1`;
turns are `mask AND n_horiz == 1 AND n_vert == 1`. Two `ScatterND-add` histograms then sum
endpoints and turns per component. Gather into per-cell counts; three Where conditions assemble
the output channels.

**Local cost**: 154K bytes (the bundle's task364 was 157K). Grader: cost ~154K, score 13.05.
Net contribution: **+0.02 LB** (grader returned 0.01 less than predicted, almost certainly
fp16 rounding in the deep graph - the only sub of the three not exact to the hundredth).


In [ ]:
"""task364 ONNX builder.

Rule (verified 266/266): for each 4-connected component of color 3, recolor by topology:
  - 3+ endpoints (cells with exactly 1 same-color 4-neighbor) -> color 2
  - <=2 endpoints AND <=1 turns (cells with degree 2 where neighbors are NOT collinear) -> color 1
  - <=2 endpoints AND >=2 turns -> color 6

Strategy:
  1. Extract mask = ch_3, plus other channels for output preservation.
  2. Compute per-cell 4-neighbor presence: up, down, left, right (via Pad+Slice shifts).
  3. deg_4 = up + down + left + right.
     n_horiz = left + right.
     n_vert = up + down.
     is_endpoint = mask AND (deg_4 == 1).
     is_turn = mask AND (n_horiz == 1) AND (n_vert == 1).
  4. Label propagation via 8-conn MaxPool 3x3 (verified equivalent to 4-conn components for this task).
     10 iterations covers max graph distance 10.
  5. Two histograms via ScatterND-with-add:
     hist_end[label] = sum of is_endpoint per component.
     hist_turn[label] = sum of is_turn per component.
  6. Per cell lookup: count_end = Gather(hist_end, label); count_turn = Gather(hist_turn, label).
  7. Output channels:
     ch_2 = mask AND (count_end >= 3)
     ch_6 = mask AND (count_end <= 2) AND (count_turn >= 2)
     ch_1 = mask AND (count_end <= 2) AND (count_turn <= 1)
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, helper as oh, numpy_helper as onh

ROOT = Path("/kaggle/working")
OUT_PATH = ROOT / "task364.onnx"


def fp32(name, value):
    return onh.from_array(np.array(value, dtype=np.float32), name=name)


def i64(name, value):
    return onh.from_array(np.array(value, dtype=np.int64), name=name)


def build():
    inits = []

    # Negated index initializer in fp16 for label propagation
    neg_idx = -np.arange(900, dtype=np.float16).reshape(1, 1, 30, 30)
    inits.append(onh.from_array(neg_idx, name="neg_idx"))

    # Constants
    inits.append(onh.from_array(np.array(-999.0, dtype=np.float16), name="NEG_BIG"))
    inits.append(fp32("ZERO", 0.0))
    inits.append(fp32("ONE", 1.0))
    inits.append(fp32("TWO", 2.0))
    inits.append(fp32("THREE", 3.0))

    # Slice indices for channels
    inits.append(i64("starts_ch0", [0, 0, 0, 0]))
    inits.append(i64("ends_ch0", [1, 1, 30, 30]))
    inits.append(i64("starts_ch3", [0, 3, 0, 0]))
    inits.append(i64("ends_ch3", [1, 4, 30, 30]))
    inits.append(i64("axes_all", [0, 1, 2, 3]))

    # Pad pads for shifting mask: 1 cell each side, 8 dims (N,C,H,W start, N,C,H,W end)
    inits.append(i64("pads_top", [0, 0, 1, 0, 0, 0, 0, 0]))     # pad top -> shifts content down (or equivalently, "up_neighbor" needs pad top)
    inits.append(i64("pads_bot", [0, 0, 0, 0, 0, 0, 1, 0]))     # pad bottom -> shifts content up
    inits.append(i64("pads_left", [0, 0, 0, 1, 0, 0, 0, 0]))    # pad left
    inits.append(i64("pads_right", [0, 0, 0, 0, 0, 0, 0, 1]))   # pad right

    # Slice indices to crop back to [1,1,30,30] after padding
    inits.append(i64("starts_crop_top", [0, 0, 0, 0]))
    inits.append(i64("ends_crop_top", [1, 1, 30, 30]))      # take rows 0-29 (drops bottom row)
    inits.append(i64("starts_crop_bot", [0, 0, 1, 0]))
    inits.append(i64("ends_crop_bot", [1, 1, 31, 30]))      # take rows 1-30 (drops top row)
    inits.append(i64("starts_crop_left", [0, 0, 0, 0]))
    inits.append(i64("ends_crop_left", [1, 1, 30, 30]))     # take cols 0-29
    inits.append(i64("starts_crop_right", [0, 0, 0, 1]))
    inits.append(i64("ends_crop_right", [1, 1, 30, 31]))    # take cols 1-30

    # For histogram
    inits.append(i64("hist_shape", [1000]))
    inits.append(i64("axes_unsqueeze_last", [4]))

    # Static zero channel for output padding
    inits.append(onh.from_array(np.zeros((1, 1, 30, 30), dtype=np.float32), name="zero_ch"))

    nodes = []

    # ---- Step 1: extract channels ----
    nodes.append(oh.make_node("Slice", ["input", "starts_ch0", "ends_ch0", "axes_all"], ["ch0"], name="slice_ch0"))
    nodes.append(oh.make_node("Slice", ["input", "starts_ch3", "ends_ch3", "axes_all"], ["mask"], name="slice_ch3"))
    nodes.append(oh.make_node("Greater", ["mask", "ZERO"], ["mask_bool"], name="mask_bool"))

    # ---- Step 2: compute neighbor presence (up, down, left, right) ----
    # up_nbr[r,c] = mask[r-1,c]: shift content DOWN by 1 row = pad top with 0, take rows 0-29
    nodes.append(oh.make_node("Pad", ["mask", "pads_top", "ZERO"], ["mask_padded_t"], mode="constant", name="pad_t"))
    nodes.append(oh.make_node("Slice", ["mask_padded_t", "starts_crop_top", "ends_crop_top", "axes_all"], ["up_nbr"], name="slice_up"))

    # down_nbr[r,c] = mask[r+1,c]: pad bottom, take rows 1-30
    nodes.append(oh.make_node("Pad", ["mask", "pads_bot", "ZERO"], ["mask_padded_b"], mode="constant", name="pad_b"))
    nodes.append(oh.make_node("Slice", ["mask_padded_b", "starts_crop_bot", "ends_crop_bot", "axes_all"], ["down_nbr"], name="slice_down"))

    # left_nbr[r,c] = mask[r,c-1]: pad left, take cols 0-29
    nodes.append(oh.make_node("Pad", ["mask", "pads_left", "ZERO"], ["mask_padded_l"], mode="constant", name="pad_l"))
    nodes.append(oh.make_node("Slice", ["mask_padded_l", "starts_crop_left", "ends_crop_left", "axes_all"], ["left_nbr"], name="slice_left"))

    # right_nbr[r,c] = mask[r,c+1]: pad right, take cols 1-30
    nodes.append(oh.make_node("Pad", ["mask", "pads_right", "ZERO"], ["mask_padded_r"], mode="constant", name="pad_r"))
    nodes.append(oh.make_node("Slice", ["mask_padded_r", "starts_crop_right", "ends_crop_right", "axes_all"], ["right_nbr"], name="slice_right"))

    # n_vert = up + down; n_horiz = left + right
    nodes.append(oh.make_node("Sum", ["up_nbr", "down_nbr"], ["n_vert"], name="sum_vert"))
    nodes.append(oh.make_node("Sum", ["left_nbr", "right_nbr"], ["n_horiz"], name="sum_horiz"))
    # deg_4 = n_vert + n_horiz
    nodes.append(oh.make_node("Sum", ["n_vert", "n_horiz"], ["deg_4"], name="sum_deg"))

    # is_endpoint = mask AND (deg_4 == 1)
    nodes.append(oh.make_node("Equal", ["deg_4", "ONE"], ["deg_eq_1"], name="deg_eq_1"))
    nodes.append(oh.make_node("And", ["mask_bool", "deg_eq_1"], ["is_end_bool"], name="is_end"))
    nodes.append(oh.make_node("Cast", ["is_end_bool"], ["is_end"], to=TensorProto.FLOAT, name="cast_is_end"))

    # is_turn = mask AND (n_horiz == 1) AND (n_vert == 1)
    nodes.append(oh.make_node("Equal", ["n_horiz", "ONE"], ["nh_eq_1"], name="nh_eq_1"))
    nodes.append(oh.make_node("Equal", ["n_vert", "ONE"], ["nv_eq_1"], name="nv_eq_1"))
    nodes.append(oh.make_node("And", ["nh_eq_1", "nv_eq_1"], ["turn_pre"], name="turn_pre"))
    nodes.append(oh.make_node("And", ["mask_bool", "turn_pre"], ["is_turn_bool"], name="is_turn"))
    nodes.append(oh.make_node("Cast", ["is_turn_bool"], ["is_turn"], to=TensorProto.FLOAT, name="cast_is_turn"))

    # ---- Step 3: label propagation (10 iter 3x3 MaxPool 8-conn) ----
    nodes.append(oh.make_node("Where", ["mask_bool", "neg_idx", "NEG_BIG"], ["nlabels_0"], name="nlabels_init"))
    prev = "nlabels_0"
    for it in range(10):
        pooled = f"pooled_{it}"
        nodes.append(oh.make_node("MaxPool", [prev], [pooled],
                                  kernel_shape=[3, 3], strides=[1, 1], pads=[1, 1, 1, 1],
                                  name=f"pool_{it}"))
        nxt = f"nlabels_{it+1}"
        nodes.append(oh.make_node("Where", ["mask_bool", pooled, "NEG_BIG"], [nxt], name=f"remask_{it}"))
        prev = nxt
    nodes.append(oh.make_node("Neg", [prev], ["labels"], name="labels_final"))

    # ---- Step 4: cast labels to int64 ----
    nodes.append(oh.make_node("Cast", ["labels"], ["labels_i64"], to=TensorProto.INT64, name="cast_labels"))
    nodes.append(oh.make_node("Unsqueeze", ["labels_i64", "axes_unsqueeze_last"], ["labels_idx"], name="unsq_idx"))

    # ---- Step 5: two histograms ----
    nodes.append(oh.make_node("ConstantOfShape", ["hist_shape"], ["hist_init"],
                              value=onh.from_array(np.array([0.0], dtype=np.float32)),
                              name="hist_init"))
    nodes.append(oh.make_node("ScatterND", ["hist_init", "labels_idx", "is_end"],
                              ["hist_end"], reduction="add", name="scatter_end"))
    nodes.append(oh.make_node("ScatterND", ["hist_init", "labels_idx", "is_turn"],
                              ["hist_turn"], reduction="add", name="scatter_turn"))

    # ---- Step 6: Gather per-cell counts ----
    nodes.append(oh.make_node("Gather", ["hist_end", "labels_i64"], ["count_end"], axis=0, name="gather_end"))
    nodes.append(oh.make_node("Gather", ["hist_turn", "labels_i64"], ["count_turn"], axis=0, name="gather_turn"))

    # ---- Step 7: build output channels ----
    # ch_2 = mask AND (count_end >= 3) <=> (count_end > 2)
    nodes.append(oh.make_node("Greater", ["count_end", "TWO"], ["end_ge3"], name="end_ge3"))
    nodes.append(oh.make_node("And", ["mask_bool", "end_ge3"], ["is_2_bool"], name="is_2"))
    nodes.append(oh.make_node("Cast", ["is_2_bool"], ["ch2_out"], to=TensorProto.FLOAT, name="cast_ch2"))

    # ch_6 = mask AND NOT(count_end >= 3) AND (count_turn >= 2) <=> (count_end <= 2) AND (count_turn > 1)
    nodes.append(oh.make_node("Greater", ["count_turn", "ONE"], ["turn_ge2"], name="turn_ge2"))
    nodes.append(oh.make_node("Not", ["end_ge3"], ["end_le2"], name="end_le2"))
    nodes.append(oh.make_node("And", ["mask_bool", "end_le2"], ["mask_no2"], name="mask_no2"))
    nodes.append(oh.make_node("And", ["mask_no2", "turn_ge2"], ["is_6_bool"], name="is_6"))
    nodes.append(oh.make_node("Cast", ["is_6_bool"], ["ch6_out"], to=TensorProto.FLOAT, name="cast_ch6"))

    # ch_1 = mask AND NOT(is_2) AND NOT(is_6)
    nodes.append(oh.make_node("Not", ["turn_ge2"], ["turn_le1"], name="turn_le1"))
    nodes.append(oh.make_node("And", ["mask_no2", "turn_le1"], ["is_1_bool"], name="is_1"))
    nodes.append(oh.make_node("Cast", ["is_1_bool"], ["ch1_out"], to=TensorProto.FLOAT, name="cast_ch1"))

    # Concat 10 channels: [ch0, ch1, ch2, zero, zero, zero, ch6, zero, zero, zero]
    concat_inputs = ["ch0", "ch1_out", "ch2_out", "zero_ch", "zero_ch", "zero_ch",
                     "ch6_out", "zero_ch", "zero_ch", "zero_ch"]
    nodes.append(oh.make_node("Concat", concat_inputs, ["output"], axis=1, name="concat_out"))

    input_vi = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
    output_vi = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])
    graph = oh.make_graph(nodes=nodes, name="task364",
                          inputs=[input_vi], outputs=[output_vi],
                          initializer=inits)
    opset = oh.make_opsetid("", 17)
    model = oh.make_model(graph, opset_imports=[opset], ir_version=8)
    model.producer_name = ""
    onnx.checker.check_model(model, full_check=True)
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    onnx.save(model, str(OUT_PATH))
    return model

_t364 = build()
print(f'task364 built: {OUT_PATH.stat().st_size} bytes, {len(_t364.graph.node)} nodes')


In [ ]:
_t364_examples = load_examples(364)
_t364_ex = _t364_examples['train'] + _t364_examples['test'] + _t364_examples.get('arc-gen', [])
_t364_pass, _t364_total = verify(WORKING / 'task364.onnx', _t364_ex)
_t364_cost, _t364_score, _, _ = cost_and_score(WORKING / 'task364.onnx', _t364_ex)
print(f'task364: verify {_t364_pass}/{_t364_total}, cost {_t364_cost}, predicted score {_t364_score:.3f}')


## Previous techniques still recommended reading

The earlier versions of this notebook documented three canonical fusion patterns that match
the grader's cost calculation to the hundredth:

- **Pattern 1**: ReduceSum-chain fusion (eliminate the intermediate of two consecutive ReduceSum
  nodes by merging axes)
- **Pattern 2**: Cast-chain collapse (route past redundant `Cast(x,fp16) -> Cast(fp16, fp32)` pairs)
- **Pattern 3**: boolean-reduction dtype narrowing (`Cast(bool,fp) -> ReduceSum -> Greater(0)`
  rewrites as `Cast(bool,u8) -> ReduceMax -> Cast(u8,bool)` with equal node count and a much
  smaller Cast intermediate)

Those rewrites scored +4.22 LB on top of an afr1ste 5689 anchor. The 6029 bundle absorbs them
indirectly through kojimar's `gemma4_block` overrides, so re-running the scanner on the 6029
base finds zero additional candidates (the targets are already fused). The scanner code is
still archived in the notebook's earlier versions on Kaggle if anyone wants it for a different
anchor.

Two boundary results from that work that are still worth flagging:
- **Or-tree restructure on task158** and **And-fusion across the bundle**: passed verification
  locally but each scored 0 LB on the grader. The empirical rule is that rewrites which REMOVE
  nodes or NARROW dtypes within the existing op vocabulary match the grader exactly; rewrites
  which INTRODUCE novel op chains (Or, And) that did not exist in the original graph can pass
  local verification but get 0 LB credit on the grader.


## Building the submission

Three hand-built ONNX files override the corresponding tasks in the jsrdcht 6029 bundle. All
other 397 tasks are taken verbatim. The result is a `submission.zip` with 400 ONNX files.


In [ ]:
OVERRIDES = {
    277: WORKING / 'task277.onnx',
    330: WORKING / 'task330.onnx',
    364: WORKING / 'task364.onnx',
}

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for tid in range(1, NUM_TASKS + 1):
        name = f'task{tid:03d}.onnx'
        if tid in OVERRIDES:
            data = OVERRIDES[tid].read_bytes()
        else:
            data = (jsrdcht_dir / name).read_bytes()
        zf.writestr(name, data)

with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    names = sorted(zf.namelist())
print('submission built')
print('  hand-built tasks :', sorted(OVERRIDES))
print('  total ONNX       :', len(names))
print('  zip size (bytes) :', OUTPUT_ZIP.stat().st_size)


## Current manual-only v205 output

The original notebook cells above rebuild the 6042.85 public baseline. The published output for this version is the clean v205 manual rewrite bundle, copied from the attached public dataset at the end so Kaggle scores this version as the manual-only current state.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile

manual_dataset = Path('/kaggle/input/neurogolf-manual-rewrites-v205')
manual_zip = manual_dataset / 'submission.zip'
output_zip = Path('/kaggle/working/submission.zip')

if manual_zip.exists():
    shutil.copy2(manual_zip, output_zip)
else:
    # Kaggle may expose uploaded zip contents as extracted files under /kaggle/input.
    task_files = sorted(
        p for p in manual_dataset.rglob('task*.onnx')
        if p.name.startswith('task') and p.name.endswith('.onnx')
    )
    assert len(task_files) == 400, (
        f'Missing manual-only v205 artifact: {manual_zip}; '
        f'found {len(task_files)} extracted ONNX files under {manual_dataset}'
    )
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for path in task_files:
            zf.write(path, arcname=path.name)

with zipfile.ZipFile(output_zip) as zf:
    manual_only_names = sorted(zf.namelist())

assert len(manual_only_names) == 400, len(manual_only_names)
assert manual_only_names[0] == 'task001.onnx' and manual_only_names[-1] == 'task400.onnx'
assert all(name.startswith('task') and name.endswith('.onnx') for name in manual_only_names)

sha256 = hashlib.sha256(output_zip.read_bytes()).hexdigest()
print('wrote', output_zip)
print('onnx files:', len(manual_only_names))
print('zip bytes:', output_zip.stat().st_size)
print('sha256:', sha256)
print('expected public LB for this manual-only artifact: 6154.71')


## Acknowledgments

- [`jsrdcht/neurogolf-6029-submission-bundle`](https://www.kaggle.com/datasets/jsrdcht/neurogolf-6029-submission-bundle):
  the 400-task ONNX bundle this notebook uses as the anchor for 397 of 400 tasks.
- [`octaviograu/neurogolf-2026-block-lb-drilling-5740-30`](https://www.kaggle.com/code/octaviograu/neurogolf-2026-block-lb-drilling-5740-30):
  earlier notebook documenting the recursive block-LB drilling protocol that established the
  5740.30 ceiling for the pure public-bundle approach.
- Previous versions of THIS notebook documented the three canonical fusion patterns
  (ReduceSum-chain fusion, Cast-chain collapse, boolean-reduction dtype narrowing) that gave
  +4.22 LB on the afr1ste 5689 anchor. Those rewrites are absorbed indirectly into the 6029
  bundle through kojimar's intermediate overrides.
- Apache-2.0. The ONNX builder functions can be lifted into any other NeuroGolf notebook that
  needs a per-task hand-build for connected-component recoloring, size-threshold recoloring,
  or topology-based recoloring.
